# 🚀 Boosting Methods — Applied on Real Dataset
### Breast Cancer Wisconsin Dataset (Kaggle-style Classification Problem)

---

**Is notebook mein hum boosting methods ko ek real-world dataset pe apply karenge.**

**Structure:**
1. 📦 Data Loading & Understanding
2. 🔍 Exploratory Data Analysis (EDA)
3. 🔧 Preprocessing
4. 🌳 Baseline Model (Decision Tree)
5. 💪 AdaBoost — Theory Recap + Full Implementation
6. 📈 Gradient Boosting — Theory Recap + Full Implementation
7. ⚖️ Final Model Comparison
8. 🏆 Capstone Exercise

---

**Dataset:** Breast Cancer Wisconsin — 569 samples, 30 features, Binary classification (Malignant vs Benign)

> 🔑 **Yeh dataset Kaggle pe bhi available hai aur ek doctor ke liye real-life mein important hai — kya tumor cancerous hai ya nahi?**

---
## 📦 Section 1: Data Loading & Understanding

**Naye Keywords:**
- **`load_breast_cancer()`** — sklearn ka built-in dataset, same as Kaggle's Breast Cancer dataset
- **`DESCR`** — dataset description — features kya hain, kahan se aaya data
- **`target_names`** — class labels (0 = malignant/cancerous, 1 = benign/not cancerous)
- **`feature_names`** — column names (mean radius, mean texture, etc.)
- **`pd.DataFrame()`** — numpy array ko table format mein convert karna

In [ ]:
# --- Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print("✅ Libraries loaded!")

In [ ]:
# --- Load Dataset ---
data = load_breast_cancer()

# DataFrame banao
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Dataset shape:", df.shape)
print("\nClass labels:", data.target_names)   # ['malignant' 'benign']
print("\nClass distribution:")
print(df['target'].value_counts())
print("  0 = Malignant (cancerous)")
print("  1 = Benign (not cancerous)")

In [ ]:
# --- First Look ---
df.head(3)

In [ ]:
# --- Basic Info ---
print("Columns:", df.shape[1])
print("\nMissing values:")
print(df.isnull().sum().sum(), "total missing values")
print("\nData types:")
print(df.dtypes.value_counts())

### 🎯 Humne Kya Seekha?
- Dataset mein **569 patients** hain, **30 medical features** hain (tumor ka size, texture, symmetry, etc.)
- **212 Malignant** (cancerous) aur **357 Benign** (not cancerous)
- Koi missing values nahi — clean dataset!

### 🌍 Real Life Mein Kaise Use Hoga?
Hospitals mein yeh model automatically predict kar sakta hai ki ek tumor cancerous hai ya nahi, sirf medical measurements se. Doctors ko har bar manually decide nahi karna padega.

---
## 🔍 Section 2: Exploratory Data Analysis (EDA)

**Naye Keywords:**
- **EDA (Exploratory Data Analysis)** — data ko samajhna before modeling
- **`groupby().mean()`** — group ke hisaab se average nikalna
- **`corr()`** — features ke beech correlation matrix
- **`sns.heatmap()`** — correlation ka visual display
- **Class imbalance** — ek class doosre se zyada hai toh model biased ho sakta hai

In [ ]:
# --- Class Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
counts = df['target'].value_counts()
axes[0].bar(['Malignant (0)', 'Benign (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Malignant', 'Benign'],
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.suptitle('Target Distribution', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# --- Feature Comparison by Class ---
# Top 6 most important features (mean radius, texture, perimeter, area, smoothness, compactness)
top_features = ['mean radius', 'mean texture', 'mean perimeter',
                'mean area', 'mean smoothness', 'mean compactness']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    benign = df[df['target'] == 1][feat]
    malignant = df[df['target'] == 0][feat]
    
    axes[i].hist(malignant, bins=25, alpha=0.7, color='#e74c3c', label='Malignant')
    axes[i].hist(benign, bins=25, alpha=0.7, color='#2ecc71', label='Benign')
    axes[i].set_title(feat.title())
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distribution: Malignant vs Benign', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Average feature values by class ---
group_means = df.groupby('target')[top_features].mean()
group_means.index = ['Malignant', 'Benign']
print("Average values by class:")
group_means.round(2)

In [ ]:
# --- Correlation Heatmap (top 10 features only for readability) ---
top10 = df[top_features + ['target']]

plt.figure(figsize=(8, 6))
corr = top10.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Top Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 3 features most correlated with target:")
print(corr['target'].abs().sort_values(ascending=False)[1:4])

### 🎯 EDA Se Kya Seekha?
- **Malignant tumors** ka mean radius, perimeter, area **zyada hota hai** benign se
- **mean radius** aur **mean perimeter** target ke saath sabse zyada correlated hain
- Dataset thoda **imbalanced** hai (37% malignant, 63% benign) — lekin itna nahi ki kuch special karna pade

### 🌍 Real Life Mein?
Doctors bhi historically dekha karte the ki bade tumors zyada dangerous hote hain — ML model yahi pattern seekhta hai!

---
## 🔧 Section 3: Preprocessing

**Naye Keywords:**
- **`train_test_split(stratify=y)`** — class proportion maintain karte hue split karna
- **`StandardScaler`** — features ko same scale pe laana (mean=0, std=1)
- **Why scale?** — Distance-based models ko scale zaroori hai, tree models ko nahi, lekin boosting gradient use karta hai toh scaling helpful ho sakti hai

In [ ]:
# --- Features aur Target alag karo ---
X = df.drop('target', axis=1)
y = df['target']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# --- Train-Test Split (stratified) ---
# stratify=y means: train aur test dono mein same class proportion rahega
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
# --- Feature Scaling ---
# Note: Decision Tree aur RandomForest ko scaling zaroori nahi
# Lekin Gradient Boosting gradient use karta hai — scaling help kar sakti hai
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit sirf train pe
X_test_scaled = scaler.transform(X_test)         # transform dono pe

print("Scaling done!")
print("Feature mean (before):", X_train.mean().mean().round(3))
print("Feature mean (after scaling):", X_train_scaled.mean().round(6))  # ~0
print("Feature std (after scaling):", X_train_scaled.std().round(6))   # ~1

### 🎯 Preprocessing Se Kya Seekha?
- **Stratified split** ensure karta hai ki train aur test mein same class ratio ho
- **StandardScaler** features ko normalize karta hai — sab ek hi scale pe aate hain
- Tree models ko scaling chahiye nahi, lekin Gradient Boosting ke saath scaling consistent results deta hai

### 🌍 Real Life Mein?
Jab medical data hota hai toh class imbalance common hota hai (healthy >> sick). Stratified split se evaluation fair rehta hai.

---
## 🌳 Section 4: Baseline Model — Decision Tree

**Baseline model** ek simple model hota hai jiske against hum apna boosted model compare karte hain.

**Keyword:**
- **Baseline** — starting point. Agar boosting model baseline se better nahi, toh kuch galat hai.
- **`cross_val_score(cv=5)`** — 5-fold cross validation, reliable accuracy estimate

In [ ]:
# --- Baseline Decision Tree ---
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)

dt_train_acc = dt.score(X_train_scaled, y_train)
dt_test_acc = dt.score(X_test_scaled, y_test)
dt_cv = cross_val_score(dt, X_train_scaled, y_train, cv=5)

print("=" * 40)
print("BASELINE — Decision Tree")
print("=" * 40)
print(f"Train Accuracy : {dt_train_acc:.4f}")
print(f"Test Accuracy  : {dt_test_acc:.4f}")
print(f"CV Mean Acc    : {dt_cv.mean():.4f} ± {dt_cv.std():.4f}")

if dt_train_acc > dt_test_acc + 0.05:
    print("⚠️  Overfitting detected! Train >> Test")
else:
    print("✅ Generalization OK")

In [ ]:
# --- Classification Report ---
dt_pred = dt.predict(X_test_scaled)
print(classification_report(y_test, dt_pred, target_names=['Malignant', 'Benign']))

---
## 💪 Section 5: AdaBoost — Full Implementation

**Theory Recap (Quick):**
- AdaBoost = Adaptive Boosting
- Har round mein galat classified samples ka weight badhta hai
- Weak learners (depth=1 trees) ek ke baad ek train hote hain
- Final prediction = weighted vote of all weak learners

**Naye Keywords:**
- **`AdaBoostClassifier`** — sklearn ka AdaBoost
- **`algorithm='SAMME'`** — discrete AdaBoost (sklearn 1.4+ default)
- **`learning_rate`** — har weak learner ka contribution kitna ho
- **`staged_score()`** — har boosting round ke baad accuracy track karna

In [ ]:
# --- AdaBoost Model ---
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision Stump
    n_estimators=100,
    learning_rate=1.0,
    algorithm='SAMME',    # Discrete AdaBoost (stable in sklearn 1.4+)
    random_state=42
)

ada.fit(X_train_scaled, y_train)

ada_train = ada.score(X_train_scaled, y_train)
ada_test = ada.score(X_test_scaled, y_test)
ada_cv = cross_val_score(ada, X_train_scaled, y_train, cv=5)

print("=" * 40)
print("AdaBoost Results")
print("=" * 40)
print(f"Train Accuracy : {ada_train:.4f}")
print(f"Test Accuracy  : {ada_test:.4f}")
print(f"CV Mean Acc    : {ada_cv.mean():.4f} ± {ada_cv.std():.4f}")
print(f"\nImprovement over DT: +{ada_test - dt_test_acc:.4f}")

In [ ]:
# --- Confusion Matrix: OLD WAY vs NEW WAY ---
ada_pred = ada.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---- OLD WAY ----
cm = confusion_matrix(y_test, ada_pred)
axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('OLD WAY: confusion_matrix + imshow')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Malignant', 'Benign'])
axes[0].set_yticklabels(['Malignant', 'Benign'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

# ---- NEW WAY ----
disp = ConfusionMatrixDisplay.from_estimator(
    ada, X_test_scaled, y_test,
    display_labels=['Malignant', 'Benign'],
    cmap='Blues', ax=axes[1]
)
axes[1].set_title('NEW WAY: ConfusionMatrixDisplay (1 line!)')

plt.suptitle('AdaBoost — Confusion Matrix Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- AdaBoost Classification Report ---
print("AdaBoost Classification Report:")
print(classification_report(y_test, ada_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# --- Learning Curve: Accuracy vs n_estimators ---
# staged_score() batata hai ki har boosting round ke baad accuracy kya thi
train_scores = list(ada.staged_score(X_train_scaled, y_train))
test_scores = list(ada.staged_score(X_test_scaled, y_test))

plt.figure(figsize=(9, 4))
plt.plot(range(1, 101), train_scores, label='Train', color='#3498db', linewidth=2)
plt.plot(range(1, 101), test_scores, label='Test', color='#e74c3c', linewidth=2)
plt.axhline(dt_test_acc, color='gray', linestyle='--', label=f'DT Baseline ({dt_test_acc:.3f})')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('AdaBoost: Accuracy vs n_estimators', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- AdaBoost Feature Importance ---
feat_imp = pd.Series(ada.feature_importances_, index=X.columns)
top15 = feat_imp.nlargest(15).sort_values()

plt.figure(figsize=(8, 6))
top15.plot(kind='barh', color='#9b59b6')
plt.xlabel('Importance')
plt.title('AdaBoost — Top 15 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 Features:")
print(feat_imp.nlargest(5).round(4))

### 🎯 AdaBoost Se Kya Seekha?
- Simple Decision Stump (depth=1) trees ko combine karke AdaBoost bahut better accuracy deta hai
- **`staged_score()`** se hum dekh sakte hain ki accuracy round-by-round kaise improve hoti hai
- Feature importances batate hain ki kaunsa medical measurement sabse useful hai cancer detect karne ke liye

### 🌍 Real Life Mein?
Medical diagnosis mein AdaBoost use hota hai — har round mein wo difficult (misclassified) patients pe zyada focus karta hai, exactly jaisa ek experienced doctor karta hai!

---
## 📈 Section 6: Gradient Boosting — Full Implementation

**Theory Recap (Quick):**
- Gradient Boosting = train each tree on the **residuals** (errors) of previous model
- Learning rate (ν) controls how much each tree contributes
- Low learning rate + more trees = usually better performance

**Naye Keywords:**
- **`GradientBoostingClassifier`** — sklearn ka Gradient Boosting
- **`subsample`** — har tree ko kitne percent data pe train karna (< 1.0 = Stochastic GB)
- **`max_features`** — har split pe kitne features consider karne
- **`min_samples_split`** — ek node split hone ke liye minimum samples
- **Stochastic Gradient Boosting** — subsample < 1.0 — faster + sometimes better (reduces overfitting)

In [ ]:
# --- Gradient Boosting Model ---
gbm = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,       # Stochastic GB — 80% data per tree
    max_features='sqrt', # sqrt(30) ≈ 5 features per split
    random_state=42
)

gbm.fit(X_train_scaled, y_train)

gbm_train = gbm.score(X_train_scaled, y_train)
gbm_test = gbm.score(X_test_scaled, y_test)
gbm_cv = cross_val_score(gbm, X_train_scaled, y_train, cv=5)

print("=" * 40)
print("Gradient Boosting Results")
print("=" * 40)
print(f"Train Accuracy : {gbm_train:.4f}")
print(f"Test Accuracy  : {gbm_test:.4f}")
print(f"CV Mean Acc    : {gbm_cv.mean():.4f} ± {gbm_cv.std():.4f}")
print(f"\nImprovement over DT: +{gbm_test - dt_test_acc:.4f}")

In [ ]:
# --- Confusion Matrix (New Way) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AdaBoost
ConfusionMatrixDisplay.from_estimator(
    ada, X_test_scaled, y_test,
    display_labels=['Malignant', 'Benign'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'AdaBoost (Test Acc: {ada_test:.3f})')

# GBM
ConfusionMatrixDisplay.from_estimator(
    gbm, X_test_scaled, y_test,
    display_labels=['Malignant', 'Benign'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title(f'Gradient Boosting (Test Acc: {gbm_test:.3f})')

plt.suptitle('AdaBoost vs GBM — Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- GBM Classification Report ---
gbm_pred = gbm.predict(X_test_scaled)
print("Gradient Boosting Classification Report:")
print(classification_report(y_test, gbm_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# --- Learning Curve: GBM staged_score ---
gbm_train_scores = list(gbm.staged_score(X_train_scaled, y_train))
gbm_test_scores = list(gbm.staged_score(X_test_scaled, y_test))

plt.figure(figsize=(9, 4))
plt.plot(range(1, 101), gbm_train_scores, label='Train', color='#27ae60', linewidth=2)
plt.plot(range(1, 101), gbm_test_scores, label='Test', color='#e74c3c', linewidth=2)
plt.axhline(dt_test_acc, color='gray', linestyle='--', label=f'DT Baseline ({dt_test_acc:.3f})')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting: Accuracy vs n_estimators', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Learning Rate vs n_estimators Tradeoff ---
# Kitna effect hota hai learning rate ka?
lr_values = [0.001, 0.01, 0.1, 0.5, 1.0]
n_values = [50, 100, 200]

results = []
for lr in lr_values:
    row = {'learning_rate': lr}
    for n in n_values:
        model = GradientBoostingClassifier(
            learning_rate=lr, n_estimators=n,
            max_depth=3, random_state=42
        )
        model.fit(X_train_scaled, y_train)
        row[f'n={n}'] = round(model.score(X_test_scaled, y_test), 4)
    results.append(row)

df_results = pd.DataFrame(results).set_index('learning_rate')
print("Test Accuracy: Learning Rate vs n_estimators")
print(df_results)

In [ ]:
# --- GBM Feature Importance ---
gbm_feat = pd.Series(gbm.feature_importances_, index=X.columns)
top15_gbm = gbm_feat.nlargest(15).sort_values()

plt.figure(figsize=(8, 6))
top15_gbm.plot(kind='barh', color='#27ae60')
plt.xlabel('Importance')
plt.title('Gradient Boosting — Top 15 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

### 🎯 Gradient Boosting Se Kya Seekha?
- GBM har tree ko **residuals pe train karta hai** — galtiyan dheere dheere kam hoti hain
- **`subsample=0.8`** (Stochastic GB) overfitting reduce karta hai
- **Low learning rate + zyada trees** = usually better generalization

### 🌍 Real Life Mein?
Credit card fraud detection, insurance claim prediction, medical diagnosis — GBM industry mein bohot common hai kyunki wo complex relationships bhi seekh sakta hai!

---
## ⚖️ Section 7: Final Model Comparison

Sab models ko ek saath compare karte hain — Decision Tree, Random Forest, AdaBoost, Gradient Boosting

In [ ]:
# --- Random Forest (for reference) ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_test = rf.score(X_test_scaled, y_test)
rf_cv = cross_val_score(rf, X_train_scaled, y_train, cv=5)

print("Random Forest:", rf_test)

In [ ]:
# --- Comparison Table ---
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'AdaBoost', 'Gradient Boosting'],
    'Test Accuracy': [dt_test_acc, rf_test, ada_test, gbm_test],
    'CV Mean': [
        dt_cv.mean(),
        rf_cv.mean(),
        ada_cv.mean(),
        gbm_cv.mean()
    ],
    'CV Std': [
        dt_cv.std(),
        rf_cv.std(),
        ada_cv.std(),
        gbm_cv.std()
    ],
    'Type': ['Single Tree', 'Bagging', 'Boosting', 'Boosting']
})

comparison = comparison.sort_values('Test Accuracy', ascending=False)
comparison[['Test Accuracy', 'CV Mean', 'CV Std']] = comparison[['Test Accuracy', 'CV Mean', 'CV Std']].round(4)
print(comparison.to_string(index=False))

In [ ]:
# --- Visual Comparison ---
models = comparison['Model'].tolist()
test_accs = comparison['Test Accuracy'].tolist()
cv_means = comparison['CV Mean'].tolist()
cv_stds = comparison['CV Std'].tolist()

colors = ['#95a5a6', '#3498db', '#9b59b6', '#27ae60']
x = np.arange(len(models))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart: Test Accuracy
bars = axes[0].bar(x, test_accs, color=colors, width=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].set_ylim(0.88, 1.01)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy Comparison')
for bar, val in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# CV Score with error bars
axes[1].bar(x, cv_means, color=colors, width=0.6, yerr=cv_stds, capsize=5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15, ha='right')
axes[1].set_ylim(0.88, 1.01)
axes[1].set_ylabel('CV Score')
axes[1].set_title('Cross-Validation Score (± Std)')

plt.suptitle('Model Comparison: Decision Tree vs Ensemble Methods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance: AdaBoost vs GBM ---
ada_top10 = pd.Series(ada.feature_importances_, index=X.columns).nlargest(10)
gbm_top10 = pd.Series(gbm.feature_importances_, index=X.columns).nlargest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ada_top10.sort_values().plot(kind='barh', ax=axes[0], color='#9b59b6')
axes[0].set_title('AdaBoost — Top 10 Features', fontweight='bold')

gbm_top10.sort_values().plot(kind='barh', ax=axes[1], color='#27ae60')
axes[1].set_title('Gradient Boosting — Top 10 Features', fontweight='bold')

plt.suptitle('Feature Importances Comparison', fontsize=13)
plt.tight_layout()
plt.show()

### 🎯 Final Comparison Se Kya Seekha?

| Model | Type | Accuracy |
|---|---|---|
| Decision Tree | Single | ✅ Baseline |
| Random Forest | Bagging | ✅✅ Better |
| AdaBoost | Boosting | ✅✅✅ Even Better |
| Gradient Boosting | Boosting | ✅✅✅✅ Best |

- Boosting methods generally bagging se better perform karte hain complex datasets pe
- AdaBoost aur GBM dono single tree se significantly better hain

### 🌍 Real Life Mein?
Industry mein Gradient Boosting (ya uske advanced versions XGBoost, LightGBM) ko prefer karte hain kyunki:
- Complex patterns seekh sakta hai
- Good with tabular data (bank data, medical data, sales data)
- Kaggle competitions mein bhi zyada use hota hai!

---
## 🔧 Section 8: Hyperparameter Tuning — GridSearchCV

**Keyword:**
- **`GridSearchCV`** — sab combinations try karke best hyperparameters dhundhna
- **`param_grid`** — dictionary of parameters to try
- **`best_params_`** — best combination jo mili
- **`best_score_`** — best CV accuracy

In [ ]:
# --- GridSearchCV for GBM ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4]
}

print("Total combinations:", 3*3*3, "models")

grid_gbm = GridSearchCV(
    GradientBoostingClassifier(subsample=0.8, random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_gbm.fit(X_train_scaled, y_train)

print("\n✅ Best Parameters:", grid_gbm.best_params_)
print(f"Best CV Score : {grid_gbm.best_score_:.4f}")
print(f"Test Accuracy : {grid_gbm.best_estimator_.score(X_test_scaled, y_test):.4f}")

In [ ]:
# --- Final Best Model Report ---
best_pred = grid_gbm.best_estimator_.predict(X_test_scaled)
print("Tuned GBM — Classification Report:")
print(classification_report(best_pred, y_test, target_names=['Malignant', 'Benign']))

# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(
    grid_gbm.best_estimator_, X_test_scaled, y_test,
    display_labels=['Malignant', 'Benign'],
    cmap='Greens'
)
plt.title('Tuned GBM — Best Model Confusion Matrix')
plt.tight_layout()
plt.show()

---
## 🏆 Section 9: Capstone Exercise — Diabetes Prediction

Ab aapki baari! Niche ek naya dataset hai — **Pima Indians Diabetes Dataset**.
Aapko boosting models apply karne hain.

**Features:** Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age  
**Target:** Outcome (0 = No Diabetes, 1 = Diabetes)

In [ ]:
# --- Diabetes Dataset Load ---
from sklearn.datasets import load_diabetes
# Note: sklearn ka load_diabetes regression hai
# Hum seaborn ka use karenge
import urllib.request
import io

# Using a simple synthetic diabetes-like dataset for the exercise
np.random.seed(42)
n = 500
diabetes_df = pd.DataFrame({
    'Pregnancies': np.random.randint(0, 15, n),
    'Glucose': np.random.randint(70, 200, n),
    'BloodPressure': np.random.randint(40, 110, n),
    'SkinThickness': np.random.randint(10, 60, n),
    'Insulin': np.random.randint(0, 300, n),
    'BMI': np.round(np.random.uniform(18, 50, n), 1),
    'DiabetesPedigreeFunction': np.round(np.random.uniform(0.08, 2.5, n), 3),
    'Age': np.random.randint(21, 81, n)
})
# Target: diabetes probability based on glucose + BMI
prob = (diabetes_df['Glucose']/200 * 0.5 +
        diabetes_df['BMI']/50 * 0.3 +
        diabetes_df['Age']/80 * 0.2)
diabetes_df['Outcome'] = (prob + np.random.normal(0, 0.1, n) > 0.5).astype(int)

print("Diabetes Dataset:")
print(diabetes_df.shape)
print(diabetes_df.head(3))
print("\nTarget:", diabetes_df['Outcome'].value_counts().to_dict())

In [ ]:
# ============================================================
# 🏋️ CAPSTONE TASK — Aapko yeh complete karna hai
# ============================================================

# TODO A: Features (X) aur target (y) separate karo
# X = ...
# y = ...

# TODO B: Train-test split karo (stratified, test_size=0.2)
# X_train, X_test, y_train, y_test = ...

# TODO C: StandardScaler lagao
# ...

# TODO D: AdaBoost model banao aur train karo
# (n_estimators=100, algorithm='SAMME')
# ada_diab = ...
# ada_diab.fit(...)

# TODO E: GradientBoostingClassifier banao aur train karo
# (n_estimators=100, learning_rate=0.1, max_depth=3)
# gbm_diab = ...
# gbm_diab.fit(...)

# TODO F: Dono models ki accuracy compare karo (test + CV)
# ...

# TODO G: Dono ke confusion matrices side by side plot karo (ConfusionMatrixDisplay)
# ...

# TODO H: Feature importance plot banao (GBM ki)
# ...

print("Capstone exercise ready. TODO A se H complete karo!")

---
## 🎓 Notebook Summary — Aaj Humne Kya Seekha?

| Concept | Matlab | Real World Use |
|---|---|---|
| Breast Cancer Dataset | 30 medical features se cancer predict karna | Hospital diagnosis systems |
| Stratified Split | Class proportion maintain karna | Imbalanced medical/financial data |
| AdaBoost | Weak learners ka weighted combination | Fraud detection, medical diagnosis |
| staged_score() | Round-by-round accuracy track karna | Model training monitoring |
| Gradient Boosting | Residuals pe training, dheere dheere improve | Kaggle competitions, fintech |
| GridSearchCV | Best hyperparameters dhundna | Production model optimization |
| ConfusionMatrixDisplay | 1 line mein confusion matrix | Quick model evaluation |
| Feature Importance | Kaunsa feature zyada useful hai | Medical feature selection |

---

### 🚀 Aage Kya Seekhein?
1. **XGBoost** — Gradient Boosting ka optimized version (faster + better)
2. **LightGBM** — Large datasets ke liye (Microsoft ka)
3. **CatBoost** — Categorical features ke saath automatically kaam karta hai
4. **Stacking** — Multiple models ko meta-learner se combine karna
5. **SHAP Values** — Model decisions explain karna (AI explainability)